### Problem 1.

In [1]:
from math import comb
import numpy as np
from scipy.stats import norm
np.random.seed(42)

In [2]:
T = [0.225, 0.262, 0.217, 0.240, 0.230, 0.229, 0.235, 0.217]
S = [0.209, 0.205, 0.196, 0.210, 0.202, 0.207, 0.224, 0.223, 0.220, 0.201]

mean_T = np.mean(T)
mean_S = np.mean(S)
var_T = np.var(T, ddof=1)
var_S = np.var(S, ddof=1)
n_T = len(T)
n_S = len(S)

se_diff = np.sqrt(var_T/n_T + var_S/n_S)
Z = (mean_T - mean_S) / se_diff
p_value = 2 * (1 - norm.cdf(abs(Z)))
Z_crit = norm.ppf(0.975)

ci_low = (mean_T - mean_S) - Z_crit * se_diff
ci_high = (mean_T - mean_S) + Z_crit * se_diff

print(f"p-value: {p_value:.4f} < 0.05 -> The average proportion of words in Twain’s essays is different from that of Snodgrass.")
print(f"95% confidence interval: ({ci_low:.4f}, {ci_high:.4f})")

p-value: 0.0002 < 0.05 -> The average proportion of words in Twain’s essays is different from that of Snodgrass.
95% confidence interval: (0.0104, 0.0339)


In [3]:
combined = np.array(T + S)
perm_diffs = []

for _ in range(10_000):
    np.random.shuffle(combined)
    perm_T = combined[:n_T]
    perm_S = combined[n_T:]
    perm_diffs.append(np.mean(perm_T) - np.mean(perm_S))

p_value = np.mean(np.abs(perm_diffs) >= abs(mean_T - mean_S))

print(f"P-value: {p_value:.4f} < 0.05")

P-value: 0.0008 < 0.05


### Problem 2.

In [4]:
placebo = (80, 45) 
p_values = []
drugs = {
    "Chlorpromazine": (75, 26),
    "Dimenhydrinate": (85, 52),
    "Pentobarbital (100 mg)": (67, 35),
    "Pentobarbital (150 mg)": (85, 37)
}

def fisher_exact_test(a, b, c=45, d=35):
    n = a + b + c + d
    row1 = a + b
    row2 = c + d
    col1 = a + c
    col2 = b + d

    def hypergeom_prob(x):
        return comb(col1, x) * comb(col2, row1 - x) / comb(n, row1)

    p_observed = hypergeom_prob(a)
    p_value = 0.0
    for i in range(max(0, row1 - col2), min(row1, col1) + 1):
        p = hypergeom_prob(i)
        if p <= p_observed:
            p_value += p
            
    odds_ratio = (a * d) / (b * c)
    return odds_ratio, p_value


print(f"{'Drug':37} {'Odds Ratio':12} {'P-value':12}")
for drug, (total, a) in drugs.items():
    b = total - a
    odds_ratio, p_value = fisher_exact_test(a, b)
    p_values.append(p_value)
    print(f"{drug:30} {odds_ratio:12.3f} {p_value:12.3f}")

Drug                                  Odds Ratio   P-value     
Chlorpromazine                        0.413        0.010
Dimenhydrinate                        1.226        0.531
Pentobarbital (100 mg)                0.851        0.740
Pentobarbital (150 mg)                0.600        0.120
